# Alexa Voice Assistant
**Author : Lekhraj Patil**

A voice-controlled assistant built from scratch using OpenAI Whisper for speech recognition, a TF-IDF + MLP neural network for intent classification, and pyttsx3 for text-to-speech. The final production app runs in Streamlit.

---

**Pipeline**
```
Microphone  -->  Whisper STT  -->  Intent Classifier  -->  Action Engine  -->  TTS Response
```

## Step 1 — Install Dependencies

In [ ]:
!pip install sounddevice scipy openai-whisper pyttsx3 torch scikit-learn pandas numpy streamlit gtts

## Step 2 — Record Audio from Microphone

I use `sounddevice` to capture voice from the microphone and `scipy` to save it as a WAV file.

In [ ]:
import sounddevice as sd
from scipy.io.wavfile import write

sd.query_devices()

In [ ]:
sd.default.device = (14, 12)

In [ ]:
def record_audio(filename="input.wav", duration=5, fs=48000):
    print("Recording...")
    recording = sd.rec(int(duration * fs), samplerate=fs, channels=1)
    sd.wait()
    write(filename, fs, recording)
    print("Saved to", filename)

record_audio()

## Step 3 — Setup OpenAI Whisper

Whisper is an open-source automatic speech recognition model released by OpenAI. It runs fully offline after the first download and supports multiple languages.

In [ ]:
import whisper

model = whisper.load_model("base")
print("Whisper loaded:", type(model).__name__)

## Step 4 — Speech to Text

Pass the recorded WAV file into Whisper and get back a text transcript.

In [ ]:
def speech_to_text(audio_path):
    result = model.transcribe(audio_path)
    text = result["text"].lower().strip()
    print("You said:", text)
    return text

speech_to_text("input.wav")

## Step 5 — Prepare Training Data

I created a dataset of labelled voice commands across 28 intents. Each row contains a prompt and its intent label. The dataset has ~4800 samples with augmentation.

In [ ]:
import pandas as pd

df = pd.read_csv("alexa_data.csv")
print("Shape:", df.shape)
df.head(10)

In [ ]:
print("Total intents:", df["intent"].nunique())
print(sorted(df["intent"].unique()))

In [ ]:
df["intent"].value_counts()

## Step 6 — Train Intent Classifier

I convert text to numbers using TF-IDF, then train a Multi-Layer Perceptron to classify which intent the user wants.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

df["prompt_clean"] = df["prompt"].str.replace(r"\s+\d+$", "", regex=True)

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df["prompt_clean"])
print("Feature matrix shape:", X.shape)

In [ ]:
from sklearn.neural_network import MLPClassifier

intent_model = MLPClassifier(hidden_layer_sizes=(50, 25), max_iter=500, random_state=42)
intent_model.fit(X, df["intent"])
print("Training accuracy:", round(intent_model.score(X, df["intent"]) * 100, 2), "%")

In [ ]:
def predict_intent(text):
    X_test = vectorizer.transform([text])
    return intent_model.predict(X_test)[0]

print(predict_intent("play some music"))
print(predict_intent("set alarm for 7 am"))
print(predict_intent("what is the news today"))
print(predict_intent("tell me a joke"))
print(predict_intent("add milk to my shopping list"))

## Step 7 — Action Engine

Based on the predicted intent, the assistant performs an action — opening a website, telling the time, returning a joke, and so on.

In [ ]:
import webbrowser
import datetime
import random
import re

shopping_list = []

def perform_action(intent, prompt=""):
    p = prompt.lower()

    if intent == "play_music":
        webbrowser.open("https://www.youtube.com/results?search_query=music")
        return "Playing music on YouTube"

    if intent == "news":
        webbrowser.open("https://news.google.com")
        return "Here are today's top headlines"

    if intent == "date_time":
        now = datetime.datetime.now()
        return now.strftime("It is %I:%M %p on %B %d, %Y")

    if intent == "jokes_fun":
        jokes = [
            "Why did the computer catch a cold? It left its Windows open!",
            "Why do programmers prefer dark mode? Light attracts bugs!",
        ]
        return random.choice(jokes)

    if intent == "weather":
        match = re.search(r"(in|of|for)\s+([a-zA-Z\s]+)", p)
        city = match.group(2).strip() if match else ""
        query = f"weather+in+{city.replace(' ', '+')}" if city else "weather+today"
        webbrowser.open(f"https://www.google.com/search?q={query}")
        return f"Showing weather for {city.title()}" if city else "Here is the forecast"

    if intent == "open_website":
        if "youtube" in p:
            webbrowser.open("https://www.youtube.com")
            return "Opening YouTube"
        if "google" in p:
            webbrowser.open("https://www.google.com")
            return "Opening Google"
        return "Which website should I open?"

    if intent == "alarm":
        match = re.search(r"(\d{1,2})(?::(\d{2}))?\s*(am|pm)?", p)
        if match:
            h = int(match.group(1))
            m = int(match.group(2) or 0)
            ap = (match.group(3) or "am").upper()
            return f"Alarm set for {h}:{m:02d} {ap}"
        return "Alarm set"

    if intent == "set_timer":
        nums = re.findall(r"\d+", p)
        mins = int(nums[0]) if nums else 1
        return f"Timer set for {mins} minutes"

    if intent == "reminder":
        return "Reminder saved"

    if intent == "shopping_list":
        item = re.sub(r"^(add|put|buy)\s+", "", p).strip()
        shopping_list.append(item)
        return f"Added {item} to your shopping list"

    if intent == "movies":
        webbrowser.open("https://www.imdb.com/chart/top/")
        return "Here are some popular movies"

    if intent == "personality":
        return "I am Alexa, built by Lekhraj Patil!"

    if intent == "general_qa":
        webbrowser.open(f"https://www.google.com/search?q={prompt.replace(' ', '+')}")
        return "Here is what I found"

    return "Sorry, I did not understand that"

print(perform_action("date_time"))
print(perform_action("jokes_fun"))

## Step 8 — Text to Speech

I use `pyttsx3` with Windows SAPI5 to convert the response text to speech.

In [ ]:
import pyttsx3

engine = pyttsx3.init("sapi5")
voices = engine.getProperty("voices")
engine.setProperty("voice", voices[1].id)
engine.setProperty("rate", 165)

def speak(text):
    print("Alexa:", text)
    engine.say(text)
    engine.runAndWait()

speak("Hello, I am Alexa. Built by Lekhraj Patil.")

## Step 9 — Run the Full Assistant

In [ ]:
def run_assistant():
    record_audio("input.wav")
    text = speech_to_text("input.wav")
    intent = predict_intent(text)
    print("Intent:", intent)
    response = perform_action(intent, text)
    speak(response)

run_assistant()

## Step 10 — Streamlit App

The full production app with browser mic input and gTTS audio playback is in `alexa.py`.

Run it locally:
```bash
streamlit run alexa.py
```

---
**Author: Lekhraj Patil**